In [1]:
import numpy as np
import pymdp
# from pymdp.inference import run_vanilla_fpi
from pymdp.algos import run_vanilla_fpi
from pymdp.utils import obj_array_uniform
from pymdp.maths import softmax, softmax_obj_arr
from pymdp.agent import Agent

In [20]:
import numpy as np

n = 4 
n_rows = 4
n_cols = 4
  # number of rows/cols
A = np.empty((n, 2, n), dtype=float)

# ---------- 1) First 2D matrix: A[:, 0, :] ----------
# First row ~0.7, others equal, column-stochastic (sum = 1)

# high = 0.7
# low = (1.0 - high) / (n - 1)  # distributes remaining mass equally

# A[:, 0, :] = low                      # fill everything with low
# A[0, 0, :] = high                     # set the first row to high


# ---------- 2) Second 2D matrix: A[:, 1, :] ----------
# Mass around the secondary diagonal (top-right to bottom-left),
# with some “noise” ±1 index away, and each column sums to 1.

main_weight  = 0.7
noise_weight = 0.1

M = np.zeros((n_rows, n_cols), dtype=float)

for col in range(n_cols):
    diag_row = (n_rows - 1) - col          # secondary diagonal position

    # main diagonal signal
    M[diag_row, col] = main_weight

    # noise above
    if diag_row - 1 >= 0:
        M[diag_row - 1, col] = noise_weight

    # noise below
    if diag_row + 1 < n_rows:
        M[diag_row + 1, col] = noise_weight

M0 = M.copy()
likelihood_lower_T = 0.9
n_obs_smell = 4
M0[:, :2] = (1 - likelihood_lower_T) * M[:, :2] + likelihood_lower_T * np.ones((n_obs_smell, 1)) / n_obs_smell

# Normalize each column to sum to 1
# M /= M.sum(axis=0, keepdims=True)
# M0 /= M.sum()
A[:, 0, :] = M0
A[:, 1, :] = M

A /= A.sum(axis=0, keepdims=True)
# A now has shape (4, 2, 4) with the desired structure
print('A with threat')
print(A[:, 1, :])

print('\n A without threat')
print(A[:, 0, :])

A with threat
[[0.         0.         0.11111111 0.875     ]
 [0.         0.11111111 0.77777778 0.125     ]
 [0.125      0.77777778 0.11111111 0.        ]
 [0.875      0.11111111 0.         0.        ]]

 A without threat
[[0.22959184 0.22727273 0.11111111 0.875     ]
 [0.22959184 0.23737374 0.77777778 0.125     ]
 [0.23979592 0.2979798  0.11111111 0.        ]
 [0.30102041 0.23737374 0.         0.        ]]


In [33]:
n_dist = 4


M_dist = np.zeros((n_dist, n_dist), dtype=float)
B_T_main_w  = 0.3
B_T_noise_w = 0.2

for col in range(n_dist):

    # Main weight position
    if col == 0:
        row = 0           # column 0 special case
    else:
        row = col - 1     # general rule

    # Add noise above (row-1), only if still upper triangular
    for i in range(n_dist):
        if row - i >= 0:
            M_dist[row - i, col] = B_T_noise_w

    # Add noise below (row+1), only if row+1 <= col (upper triangular)
    if row + 1 <= col and row + 1 < n:
        M_dist[row + 1, col] = B_T_noise_w
    
    M_dist[row, col] = B_T_main_w

# Normalize each column to sum to 1 (if desired)
M_dist /= M_dist.sum(axis=0, keepdims=True)

print(M_dist)

[[1.         0.6        0.28571429 0.22222222]
 [0.         0.4        0.42857143 0.22222222]
 [0.         0.         0.28571429 0.33333333]
 [0.         0.         0.         0.22222222]]


In [2]:
# Hidden State Factor 1 = Disease {Diseased, Not Diseased}
# Observation Modality 1 = Test {Positive, Negative}
# No transitions, no preferences => B, C not neeeded

hidden_states = ['Diseased', 'Not Diseased']
test_observations = ['Positive', 'Negative']

# Priors about hidden states => D matrix [D, ND]
# Assuming 1% prevalance 
p_disease = 0.01
D = np.array([p_disease, 1-p_disease])

D_mtx = obj_array_uniform([D.shape])
D_mtx[0] = D.copy()

D_mtx

array([array([0.01, 0.99])], dtype=object)

In [3]:
# B -> (states, states, actions)
B = np.eye(len(hidden_states))
B = B[:, :, np.newaxis] # Add one dim for actions, even tho no actions

B_mtx = obj_array_uniform([B.shape])
B_mtx[0] = B.copy()

B_mtx

array([array([[[1.],
               [0.]],

              [[0.],
               [1.]]])], dtype=object)

In [5]:
# Likelihoods => A matrix 2x2 (obs x hidden states)
# P(diseased | positive) {sensitivity/TP rate} = 0.99
# P(not diseased | positive) {FP rate} = 0.05
import numpy as np

tp_rate = 0.99 # D,P
fp_rate = 0.05 # ND, P
tn_rate = 1-fp_rate # ND, N
fn_rate = 1-tp_rate # D, N
A = np.array([[tp_rate, fp_rate], [fn_rate, tn_rate]])

A_mtx = np.array([A], dtype=object)

len(A_mtx)
A_mtx[0].dtype

dtype('O')

In [5]:
obs = [[1,0], [0,1]] # positive, negative

qs = run_vanilla_fpi(A_mtx, np.array([1, 0]), num_obs=[2], num_states=[2], prior=D_mtx)
print(qs)

[array([0.16666667, 0.83333333])]


In [8]:
agent = Agent(A=A, B=B_mtx, D=D_mtx)
agent.infer_states(np.array([0]))

array([array([0.16666667, 0.83333333])], dtype=object)